In [ ]:
import numpy as np
import cv2
from ultralytics import YOLO


class yolov8_detector:
    def __init__(self, model_path=r"d:\project\step1\week12\yolov8n.pt"):
        self.module = YOLO(model_path)

    def get_result(self, frame):
        """
        输入一帧 BGR 图像，返回 [{bbox, conf, cls_}, ...]；无目标返回 None
        """
        out_results = []
        result = self.module.predict(frame, conf=0.4, iou=0.45, verbose=False)[0]

        bboxes = result.boxes.xyxy.cpu().numpy()   # (N, 4)  x1, y1, x2, y2
        confs = result.boxes.conf.cpu().numpy()    # (N,)    置信度
        cls_idx = result.boxes.cls.cpu().numpy()   # (N,)    类别索引

        for bbox, conf, cls_ in zip(bboxes, confs, cls_idx):
            x1, y1, x2, y2 = map(int, bbox)
            out_results.append({"bbox": [x1, y1, x2, y2],
                                "conf": float(conf),
                                "cls_": int(cls_)})

        return out_results if len(out_results) > 0 else None


if __name__ == "__main__":
    # 读 car2.mp4 视频逐帧检测
    detector = yolov8_detector()
    cap = cv2.VideoCapture(r"d:\project\step1\week13\car2.mp4")

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        results = detector.get_result(frame)
        # 每隔 60 帧打印一次（约每 2 秒），避免 3411 帧全部刷屏
        if frame_idx % 60 == 0:
            n = len(results) if results else 0
            ids = ", ".join(f"cls{r['cls_']}(conf {r['conf']:.2f})" for r in results or [])
            print(f"帧 {frame_idx:>4}: {n} 个目标 -> {ids or '无目标'}")
        frame_idx += 1
    cap.release()

    print(f"\n共处理 {frame_idx} 帧")

In [ ]:
import numpy as np
import cv2


class IOU_tracker:
    """简单 IoU 多目标跟踪器：新检测与已有轨迹按 IoU 贪心匹配"""

    def __init__(self, iou_thresh=0.3, max_lost=5):
        self.yolov8_detector = yolov8_detector()
        self.tracks = []                # 活动轨迹 [{tracker_id, bbox, conf, cls_, lost}]
        self.next_id = 1                # 下一个新 ID
        self.iou_thresh = iou_thresh    # 匹配 IoU 阈值
        self.max_lost = max_lost        # 连续多少帧未匹配则删除轨迹

    @staticmethod
    def compute_iou(box1, box2):
        """计算两个 [x1, y1, x2, y2] 框的交并比 IoU"""
        x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)          # 交集面积
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = area1 + area2 - inter                       # 并集面积
        return inter / union if union > 0 else 0.0

    def generate_tracker(self, frame_data):
        """处理一帧：检测 -> IoU 匹配 -> 更新/新建/删除轨迹，返回当前帧轨迹列表"""
        new_det = self.yolov8_detector.get_result(frame_data) or []   # 无目标时为 []

        keep = []
        matched = set()

        # 1) 贪心匹配：每个新检测找一个 IoU 最大且超过阈值的老轨迹
        for det in new_det:
            best_iou, best_idx = self.iou_thresh, -1
            for j, track in enumerate(self.tracks):
                if j in matched:                            # 已被占用的轨迹跳过
                    continue
                iou = self.compute_iou(track["bbox"], det["bbox"])
                if iou > best_iou:
                    best_iou, best_idx = iou, j

            if best_idx >= 0:                               # 匹配成功：沿用老 ID，更新框
                t = self.tracks[best_idx]
                t.update(bbox=det["bbox"], conf=det["conf"], cls_=det["cls_"], lost=0)
                keep.append(t)
                matched.add(best_idx)
            else:                                           # 新目标：分配新 ID
                keep.append({"tracker_id": self.next_id, "bbox": det["bbox"],
                             "conf": det["conf"], "cls_": det["cls_"], "lost": 0})
                self.next_id += 1

        # 2) 未匹配的老轨迹：lost 计数，超过 max_lost 帧才删除（容忍短暂遮挡/漏检）
        for j, track in enumerate(self.tracks):
            if j not in matched:
                track["lost"] += 1
                if track["lost"] <= self.max_lost:
                    keep.append(track)

        self.tracks = keep
        return keep

    def run(self, video_path):
        """读取视频逐帧跟踪并实时显示（按 q 退出）"""
        cap = cv2.VideoCapture(video_path)
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            tracks = self.generate_tracker(frame)

            # 可视化：画框 + ID
            for t in tracks:
                x1, y1, x2, y2 = t["bbox"]
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, f"ID {t['tracker_id']}", (x1, max(y1 - 5, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            cv2.imshow("IOU Tracker", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        cap.release()
        cv2.destroyAllWindows()


# ---- 演示：读取 car2.mp4 视频逐帧跟踪 ----
if __name__ == "__main__":
    tracker = IOU_tracker()
    cap = cv2.VideoCapture(r"d:\project\step1\week13\car2.mp4")

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        tracks = tracker.generate_tracker(frame)
        # 每隔 60 帧打印一次（约每 2 秒），避免 3411 帧全部刷屏
        if frame_idx % 60 == 0:
            ids = ", ".join(f"ID{t['tracker_id']}(conf {t['conf']:.2f})" for t in tracks)
            print(f"帧 {frame_idx:>4}: {len(tracks)} 个目标 -> {ids or '无目标'}")
        frame_idx += 1
    cap.release()

    print(f"\n共处理 {frame_idx} 帧，共分配 {tracker.next_id - 1} 个 ID")

    # 如需实时窗口显示，可直接用: tracker.run(r"d:\project\step1\week13\car2.mp4")

In [ ]:
# ============================================================
# 显示跟踪视频：逐帧跟踪 + 画框画ID -> 保存带标注视频 -> 内联播放
# 注意：如需真正的"实时窗口"（弹窗实时播放），改用:
#       tracker.run(r"d:\project\step1\week13\car2.mp4")  按 q 退出
# ============================================================
import cv2

tracker = IOU_tracker()
cap = cv2.VideoCapture(r"d:\project\step1\week13\car2.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 输出带跟踪标注的视频
out_path = r"d:\project\step1\week13\track_out2.mp4"
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    tracks = tracker.generate_tracker(frame)
    # 画框 + ID
    for t in tracks:
        x1, y1, x2, y2 = t["bbox"]
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"ID {t['tracker_id']}", (x1, max(y1 - 5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    writer.write(frame)
    frame_idx += 1
cap.release()
writer.release()
print(f"已生成带跟踪标注的视频: {out_path}（{frame_idx} 帧, {w}x{h}, {fps:.0f}fps）")

# 在 Notebook 内联播放
from IPython.display import Video
Video(out_path, width=480)